## 0. Load data

In [ ]:
#Change this variable to True when you want to use the training data, change to False when you want to use the testing data. This will allow you to easily switch between the two datasets without having to change the file paths in multiple places in the code.
IsTraining = True

In [54]:
import sys
sys.path.insert(0, '..')

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
from pathlib import Path

if IsTraining:
    churn_path = Path("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/datasets/resort_train.csv")
else:
    churn_path = Path("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/datasets/resort_test.csv")

churn = pd.read_csv(churn_path)

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 80)

In [55]:
#code to save the figures as high-res PNGs

IMAGES_PATH = Path() / "images" / "model_cleaning"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [56]:
#the next 5 lines define the default font sizes of graphs
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

## C. Fill in the Nulls in Promo codes with "NoPromoCode"

In [57]:
#In PromoCode column, replace the null values with "NoPromoCode"
churn['PromoCode'] = churn['PromoCode'].fillna('NoPromoCode')

# D. Fill in the nulls in Region

In [58]:
#In Region if null fill with "Unknown"
churn['Region'] = churn['Region'].fillna('Unknown')

# E. Fill in the null in the AllInclusive column

In [59]:
#fill in the null in the AllInclusive column with "0", assume if no data they are not all inclusive
churn['AllInclusive'] = churn['AllInclusive'].fillna(0)

# G. Fill the nulls in the PackageType column

In [60]:
#Fill the nulls in the PackageType column with "NoPackage"
churn['PackageType'] = churn['PackageType'].fillna('NoPackage')

# H&S. Remove as many nulls in the age and AgeGroup as possible

We can use information contained in other columns to extrapolate some of the null information

**Information from the AgeGroup to fill in the Age column will skew the data slightly, be careful if you want to make conclusions based off the age column. To start with we will drop this column**

In [61]:
churn["AgeGroup"].value_counts()

AgeGroup
Young      652
Middle     450
Minor      315
Senior     204
Elderly     44
Name: count, dtype: int64

In [62]:
#count the number of null values in AgeGroup column and  Age column
print(churn["AgeGroup"].isnull().sum())
print(churn["Age"].isnull().sum())

74
127


### Data Cleaning - Missing values - Age Group and Age

Minor      1372 1-18    10

Young      2586 19-30   24

Middle     1810 31-45   38

Senior      727 46-60   52

Elderly     176 61-79   70

we can use this data to fill in blanks in either group - I'll use the median age for the groups to fill in missing age values

In [63]:
#find null values in AgeGroup column and for each corresponding row, fill in the agegroup based on the age column. "Minor" if age >0 && <= 18, "Young" if age > 18 && <=30, "Middle" if age > 30 && <= 45, "Senior" if age > 45 && <= 60, "Elderly" if age > 60. 

mask = churn['AgeGroup'].isna() & churn['Age'].notna()

churn.loc[mask, 'AgeGroup'] = pd.cut(
    churn.loc[mask, 'Age'],
    bins=[0, 18, 30, 45, 60, float('inf')],
    labels=['Minor', 'Young', 'Middle', 'Senior', 'Elderly'],
    right=True,            # right=True means upper bound is inclusive: (lo, hi]
    include_lowest=True,  # age must be >= 0
)

print(churn["AgeGroup"].isnull().sum())
print(churn["Age"].isnull().sum())

41
127


In [64]:
#now we need to do the same thing for the Age column, find null values in Age column and for each corresponding row, fill in the age based on the agegroup column. 10 if agegroup is "Minor", 24 if agegroup is "Young", 38 if agegroup is "Middle", 52 if agegroup is "Senior", 70 if agegroup is "Elderly".
mask = churn['Age'].isna() & churn['AgeGroup'].notna()
churn.loc[mask, 'Age'] = churn.loc[mask, 'AgeGroup'].map({
    'Minor': 10,
    'Young': 24,
    'Middle': 38,
    'Senior': 52,
    'Elderly': 70
})

print(churn["AgeGroup"].isnull().sum())
print(churn["Age"].isnull().sum())

#The displayed 2 numbers should both be the same, this means that the remaining values are null in both columns and we cannot fill them in based on the other column. We may have to drop these rows later on.

41
41


In [65]:
#We have skewed the age column so let's drop the Age column
churn = churn.drop("Age", axis=1)

In [66]:
#fill in the remaining null values in the AgeGroup column with "Unknown"
churn['AgeGroup'] = churn['AgeGroup'].fillna('Unknown')

## Is there any information we can get from guests that shared a room?

In [67]:
#I want to know how many guests shared a room
#Create a new column called "SharedRoom" that is contains a 1 if there is another GuesID that shares the same Room and BookingDate, and 0 otherwise.
churn['SharedRoom'] = churn.duplicated(subset=['Room', 'BookingDate'], keep=False).astype(int)

#Count the number of guests that shared a room
num_shared = churn['SharedRoom'].sum()
print(f'Number of guests that shared a room: {num_shared}')

Number of guests that shared a room: 21


# I. Fill the nulls in VIP column

In [68]:
#In the VIP column replace the null values with "0", assume if no data they are not VIPs
churn['VIP'] = churn['VIP'].fillna(0)

# J-N. Clean the outliers from the RoomService, Dining, Retail, Spa, and Entertainment columns

In [69]:
#in the Dining, Retail, Spa, and Entertainment columns, fill in the nulls with "0", assume if no data they did not use those amenities
amenities = ['RoomService', 'Dining', 'Retail', 'Spa', 'Entertainment']
for amenity in amenities:
    churn[amenity] = churn[amenity].fillna(0)
#There are outliers in the Dining, Retail, Spa, and Entertainment columns. They all have decimal values, lets take any non-integer value and convert them to 0, assume if they have a decimal value they did not use that amenity.
for amenity in amenities:
    churn[amenity] = churn[amenity].apply(lambda x: 0 if isinstance(x, float) and not x.is_integer() else x)    


# F. Not sure what to do with the Room column

In [70]:
#drop the Room column
churn = churn.drop(columns=['Room'])

## Export the cleaned data

In [71]:
#Export the churned data to a csv file
if IsTraining:
    churn.to_csv("churn_train_cleaned.csv", index=False)
else:
    churn.to_csv("churn_test_cleaned.csv", index=False)